# Ponpare Recommendation System V1.2

Leakage-controlled learning-to-rank pipeline with event-time features, matched validation pools,
paired user bootstrap uncertainty, reproducible baselines, and bounded final refitting.


## 0. Environment & Configuration

**做什么：** 导入依赖、统一随机种子和所有实验开关。  
**为什么：** 集中配置使三种模型的比较可复现，也方便在 Kaggle 内存/时间限制下缩放实验。  
**输入：** Kaggle Python 环境。  
**输出：** 全局配置、库版本、GPU 可见性与稳定随机状态。模型训练 GPU 优先，失败时按模型回退 CPU。


In [ ]:
import gc
import importlib
import inspect
import os
import random
import re
import subprocess
import sys
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
NEGATIVE_SAMPLE_RATIO = 5
VALID_RATIO = 0.20
TOP_K = 10
USER_BATCH_SIZE = 100
USE_VISIT_FEATURES = True

TRAIN_LIGHTGBM = True
TRAIN_XGBOOST = True
TRAIN_CATBOOST = True
REFIT_BEST_ON_FULL_DATA = True

USE_GPU = True
GPU_DEVICE_ID = 0

# 控制验证候选规模；0 表示不限制验证用户数。
MAX_TUNING_USERS = 1000
MAX_VALID_USERS = 1000
VALID_CANDIDATES_PER_USER = 500  # Retrieval diagnostic only; ranker early stopping uses the full pool.
POPULAR_CANDIDATES = 100
HARD_POOL_LIMIT = 250  # Random pre-cap before distance ranking: approximate, not global nearest.
BOOTSTRAP_REPEATS = 2000

random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 80)


def ensure_import(module_name, pip_name=None):
    """Import a package and install it only when the Kaggle image does not provide it."""
    try:
        return importlib.import_module(module_name)
    except ImportError:
        package = pip_name or module_name
        print(f"Installing missing dependency: {package}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        return importlib.import_module(module_name)


lgb = ensure_import("lightgbm") if TRAIN_LIGHTGBM else None
xgb = ensure_import("xgboost") if TRAIN_XGBOOST else None
catboost = ensure_import("catboost") if TRAIN_CATBOOST else None

def detect_nvidia_gpu():
    """Detect CUDA without executing the environment-dependent `nvidia-smi` binary."""
    if not USE_GPU:
        return False

    visible = os.environ.get("CUDA_VISIBLE_DEVICES")
    if visible is not None and visible.strip() not in {"", "-1", "NoDevFiles"}:
        print("CUDA_VISIBLE_DEVICES:", visible)
        return True

    driver_path = Path("/proc/driver/nvidia/gpus")
    if driver_path.exists():
        try:
            if any(driver_path.iterdir()):
                print("NVIDIA GPU detected through /proc/driver/nvidia/gpus")
                return True
        except OSError:
            pass

    if catboost is not None and hasattr(catboost, "get_gpu_device_count"):
        try:
            count = int(catboost.get_gpu_device_count())
            if count > 0:
                print("CatBoost-visible GPU count:", count)
                return True
        except Exception:
            pass

    try:
        torch_module = importlib.util.find_spec("torch")
        if torch_module is not None:
            import torch
            if torch.cuda.is_available():
                print("PyTorch-visible GPU:", torch.cuda.get_device_name(0))
                return True
    except Exception:
        pass

    print("No visible NVIDIA GPU; rankers will use CPU.")
    return False


GPU_AVAILABLE = detect_nvidia_gpu()
print("GPU priority:", USE_GPU, "GPU available:", GPU_AVAILABLE)
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "numpy:", np.__version__)
if lgb is not None: print("lightgbm:", lgb.__version__)
if xgb is not None: print("xgboost:", xgb.__version__)
if catboost is not None:
    print("catboost:", getattr(catboost, "__version__", "unknown"))
    print("catboost module:", getattr(catboost, "__file__", "unknown"))
    print("CatBoostRanker available:", hasattr(catboost, "CatBoostRanker"))


## 1. Load Dataset

**做什么：** 扫描 `/kaggle/input`，按文件名前缀定位并读取六个必需文件。  
**为什么：** Kaggle 数据集目录名和文件名可能变化，文件也可能带 `(1)` 后缀或仍为 ZIP。  
**输入：** Ponpare CSV/CSV.ZIP 文件。  
**输出：** 用户、训练券、测试券、购买、浏览和提交模板 DataFrame。


In [ ]:
USER_ID = "USER_ID_hash"
COUPON_ID = "COUPON_ID_hash"
VIEW_COUPON_ID = "VIEW_COUPON_ID_hash"

FILE_PREFIXES = {
    "users": "user_list",
    "coupons_train": "coupon_list_train",
    "coupons_test": "coupon_list_test",
    "purchases": "coupon_detail_train",
    "visits": "coupon_visit_train",
    "submission": "sample_submission",
}


def scan_input_files(search_roots=("/kaggle/input", ".")):
    """Return readable CSV/ZIP paths under Kaggle and the current directory."""
    found = []
    for root in search_roots:
        root_path = Path(root)
        if not root_path.exists():
            continue
        for path in root_path.rglob("*"):
            if path.is_file() and (path.name.lower().endswith(".csv") or path.name.lower().endswith(".csv.zip")):
                found.append(path)
    return sorted(set(found), key=lambda p: str(p))


def find_data_file(prefix, candidates):
    """Find a CSV/CSV.ZIP by canonical prefix, accepting suffixes such as `(1)`."""
    pattern = re.compile(rf"^{re.escape(prefix)}(?:\(\d+\))?\.csv(?:\.zip)?$", re.IGNORECASE)
    matches = [p for p in candidates if pattern.match(p.name)]
    if not matches:
        # Fallback is intentionally prefix based, but still restricted to CSV artifacts.
        matches = [p for p in candidates if p.name.lower().startswith(prefix.lower())]
    if not matches:
        raise FileNotFoundError(f"Cannot locate file with prefix: {prefix}")
    matches.sort(key=lambda p: (0 if p.name.lower() in {f"{prefix}.csv", f"{prefix}.csv.zip"} else 1, len(str(p))))
    return matches[0]


def read_csv_robust(path):
    """Read a Ponpare CSV (including one-member ZIP files) with encoding fallback."""
    last_error = None
    for encoding in ("utf-8", "cp932"):
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


all_input_files = scan_input_files()
for path in all_input_files:
    if "/kaggle/input" in str(path).replace("\\", "/"):
        print(path)

DATA_PATHS = {name: find_data_file(prefix, all_input_files) for name, prefix in FILE_PREFIXES.items()}
display(pd.DataFrame({"dataset": DATA_PATHS.keys(), "path": map(str, DATA_PATHS.values())}))

users = read_csv_robust(DATA_PATHS["users"])
coupons_train_raw = read_csv_robust(DATA_PATHS["coupons_train"])
coupons_test_raw = read_csv_robust(DATA_PATHS["coupons_test"])
purchases_raw = read_csv_robust(DATA_PATHS["purchases"])
visits_raw = read_csv_robust(DATA_PATHS["visits"])
sample_submission = read_csv_robust(DATA_PATHS["submission"])


## 2. Data Overview

**做什么：** 检查形状、字段、样例、缺失和内存。  
**为什么：** Ponpare 的真实字段使用 `DISPFROM/DISPEND` 与 `ken_name`，不能依赖文档中的假设拼写。  
**输入：** 原始 DataFrame。  
**输出：** 数据字典式概览及必需字段校验结果。


In [ ]:
REQUIRED_COLUMNS = {
    "users": {USER_ID, "SEX_ID", "AGE", "PREF_NAME"},
    "coupons_train": {COUPON_ID, "GENRE_NAME", "CAPSULE_TEXT", "PRICE_RATE", "CATALOG_PRICE", "DISCOUNT_PRICE", "DISPFROM", "DISPEND", "ken_name"},
    "coupons_test": {COUPON_ID, "GENRE_NAME", "CAPSULE_TEXT", "PRICE_RATE", "CATALOG_PRICE", "DISCOUNT_PRICE", "DISPFROM", "DISPEND", "ken_name"},
    "purchases": {USER_ID, COUPON_ID, "I_DATE"},
    "visits": {USER_ID, VIEW_COUPON_ID, "I_DATE"},
    "submission": {USER_ID, "PURCHASED_COUPONS"},
}
raw_frames = {
    "users": users, "coupons_train": coupons_train_raw, "coupons_test": coupons_test_raw,
    "purchases": purchases_raw, "visits": visits_raw, "submission": sample_submission,
}

for name, frame in raw_frames.items():
    missing = REQUIRED_COLUMNS[name] - set(frame.columns)
    if missing:
        raise ValueError(f"{name} missing required columns: {sorted(missing)}")
    print(f"\n{name}: shape={frame.shape}, memory={frame.memory_usage(deep=True).sum()/1024**2:.2f} MB")
    print("columns:", frame.columns.tolist())
    display(frame.head(3))


## 3. Data Cleaning

**做什么：** 解析日期、修复类型、去除无效键并构造一致的券特征。  
**为什么：** 三种模型必须共享同一份干净数据；降精度可显著减少 Kaggle 内存占用。  
**输入：** 原始表。  
**输出：** 清洗后的 `users`、`purchases`、`visits`、`coupons_train`、`coupons_test`。


In [ ]:
def reduce_mem_usage(df, verbose=True):
    """Downcast numeric columns without mutating identifiers or datetime columns."""
    result = df.copy()
    before = result.memory_usage(deep=True).sum() / 1024**2
    protected = {USER_ID, COUPON_ID, VIEW_COUPON_ID, "positive_event_id"}
    for col in result.select_dtypes(include=["int", "float"]).columns:
        if col in protected:
            continue
        if pd.api.types.is_integer_dtype(result[col]):
            result[col] = pd.to_numeric(result[col], downcast="integer")
        else:
            result[col] = pd.to_numeric(result[col], downcast="float")
    after = result.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f"Memory: {before:.2f} MB -> {after:.2f} MB ({100*(before-after)/max(before,1e-9):.1f}% saved)")
    return result


def preprocess_coupon_features(df, split_name):
    """Clean coupon attributes and add interpretable numeric/date features."""
    out = df.copy()
    out[COUPON_ID] = out[COUPON_ID].astype("string")
    for col in ["DISPFROM", "DISPEND", "VALIDFROM", "VALIDEND"]:
        if col in out:
            out[col] = pd.to_datetime(out[col], errors="coerce")
    for col in ["PRICE_RATE", "CATALOG_PRICE", "DISCOUNT_PRICE", "DISPPERIOD", "VALIDPERIOD"]:
        if col not in out:
            out[col] = np.nan
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["discount_amount"] = (out["CATALOG_PRICE"] - out["DISCOUNT_PRICE"]).clip(lower=0)
    out["discount_ratio"] = (out["DISCOUNT_PRICE"] / out["CATALOG_PRICE"].where(out["CATALOG_PRICE"] > 0)).replace([np.inf, -np.inf], np.nan)
    out["log_catalog_price"] = np.log1p(out["CATALOG_PRICE"].clip(lower=0))
    out["log_discount_price"] = np.log1p(out["DISCOUNT_PRICE"].clip(lower=0))
    out["display_days"] = (out["DISPEND"] - out["DISPFROM"]).dt.total_seconds().div(86400).clip(lower=0)
    out["disp_year"] = out["DISPFROM"].dt.year
    out["disp_month"] = out["DISPFROM"].dt.month
    out["disp_weekday"] = out["DISPFROM"].dt.weekday
    out["disp_day"] = out["DISPFROM"].dt.day
    out["coupon_split"] = split_name
    return out.drop_duplicates(COUPON_ID).reset_index(drop=True)


users = users.copy()
users[USER_ID] = users[USER_ID].astype("string")
users["AGE"] = pd.to_numeric(users["AGE"], errors="coerce")
users["age_bucket"] = pd.cut(
    users["AGE"], bins=[-np.inf, 19, 29, 39, 49, 59, np.inf],
    labels=["<20", "20-29", "30-39", "40-49", "50-59", "60+"]
).astype("string")
users = users.drop_duplicates(USER_ID).reset_index(drop=True)

coupons_train = preprocess_coupon_features(coupons_train_raw, "train")
coupons_test = preprocess_coupon_features(coupons_test_raw, "test")

purchases = purchases_raw.copy()
purchases[USER_ID] = purchases[USER_ID].astype("string")
purchases[COUPON_ID] = purchases[COUPON_ID].astype("string")
purchases["I_DATE"] = pd.to_datetime(purchases["I_DATE"], errors="coerce")
purchases = purchases.dropna(subset=[USER_ID, COUPON_ID, "I_DATE"]).reset_index(drop=True)
purchases["positive_event_id"] = np.arange(len(purchases), dtype=np.int64)

visits = visits_raw.rename(columns={VIEW_COUPON_ID: COUPON_ID}).copy()
visits[USER_ID] = visits[USER_ID].astype("string")
visits[COUPON_ID] = visits[COUPON_ID].astype("string")
visits["I_DATE"] = pd.to_datetime(visits["I_DATE"], errors="coerce")
visits = visits.dropna(subset=[USER_ID, COUPON_ID, "I_DATE"]).reset_index(drop=True)

valid_users = set(users[USER_ID])
valid_train_coupons = set(coupons_train[COUPON_ID])
purchases = purchases[purchases[USER_ID].isin(valid_users) & purchases[COUPON_ID].isin(valid_train_coupons)].reset_index(drop=True)
visits = visits[visits[USER_ID].isin(valid_users)].reset_index(drop=True)

users = reduce_mem_usage(users)
coupons_train = reduce_mem_usage(coupons_train)
coupons_test = reduce_mem_usage(coupons_test)
purchases = reduce_mem_usage(purchases)
visits = reduce_mem_usage(visits)

print("Clean shapes:", {"users": users.shape, "coupons_train": coupons_train.shape, "coupons_test": coupons_test.shape,
                        "purchases": purchases.shape, "visits": visits.shape})
display(coupons_train.head(3))
del raw_frames, coupons_train_raw, coupons_test_raw, purchases_raw, visits_raw
gc.collect()


## 4. Build User-Coupon Interactions

**做什么：** 将每一条真实购买事件定义为正样本，并建立用户真实购买集合。  
**为什么：** 正负标签只能来自真实行为；地区匹配绝不参与标签定义。重复购买事件仍保留为独立正样本。  
**输入：** 清洗后的购买事件。  
**输出：** 全量正事件和用于防止假负样本的购买集合。


In [ ]:
positive_events = purchases[[USER_ID, COUPON_ID, "I_DATE", "positive_event_id"]].copy()
positive_events["label"] = np.int8(1)
all_purchased_sets = positive_events.groupby(USER_ID, observed=True)[COUPON_ID].agg(set).to_dict()

print("Positive purchase events:", len(positive_events))
print("Unique positive user-coupon pairs:", positive_events[[USER_ID, COUPON_ID]].drop_duplicates().shape[0])
print("Positive users:", positive_events[USER_ID].nunique())
assert positive_events["label"].eq(1).all()
display(positive_events.head())


## 5. Train / Early-stop / Holdout Split

**做什么：** 按时间将购买事件切成训练、早停调优和最终留出评估三段。  
**为什么：** 早停与最终模型比较不能使用同一批购买事件，否则报告的 MAP@10 会产生选择偏差。  
**输入：** 全量正事件。  
**输出：** `train_purchases`、`tuning_purchases`、`valid_purchases` 及两个严格时间切点。


In [ ]:
def temporal_three_way_split(events, valid_ratio=0.2):
    """Chronologically split events into train, early-stop, and untouched holdout parts."""
    ordered_dates = events["I_DATE"].sort_values().reset_index(drop=True)
    if len(ordered_dates) < 3:
        raise ValueError("At least three purchase events are required for a three-way split.")
    tuning_fraction = valid_ratio / 2
    holdout_fraction = valid_ratio / 2
    tuning_index = min(max(int(len(ordered_dates) * (1 - tuning_fraction - holdout_fraction)), 1), len(ordered_dates) - 2)
    holdout_index = min(max(int(len(ordered_dates) * (1 - holdout_fraction)), tuning_index + 1), len(ordered_dates) - 1)
    tuning_start = ordered_dates.iloc[tuning_index]
    holdout_start = ordered_dates.iloc[holdout_index]
    train = events[events["I_DATE"] < tuning_start].copy()
    tuning = events[(events["I_DATE"] >= tuning_start) & (events["I_DATE"] < holdout_start)].copy()
    holdout = events[events["I_DATE"] >= holdout_start].copy()
    if train.empty or tuning.empty or holdout.empty:
        raise ValueError("Three-way temporal split produced an empty partition.")
    return train, tuning, holdout, tuning_start, holdout_start


train_purchases, tuning_purchases, valid_purchases, EARLY_STOP_START, VALID_START = temporal_three_way_split(
    positive_events, VALID_RATIO
)
VALID_END = valid_purchases["I_DATE"].max()

split_summary = pd.DataFrame({
    "split": ["train", "early_stop", "holdout"],
    "start": [train_purchases["I_DATE"].min(), tuning_purchases["I_DATE"].min(), valid_purchases["I_DATE"].min()],
    "end": [train_purchases["I_DATE"].max(), tuning_purchases["I_DATE"].max(), valid_purchases["I_DATE"].max()],
    "users": [train_purchases[USER_ID].nunique(), tuning_purchases[USER_ID].nunique(), valid_purchases[USER_ID].nunique()],
    "positive_events": [len(train_purchases), len(tuning_purchases), len(valid_purchases)],
    "unique_positive_pairs": [
        train_purchases[[USER_ID, COUPON_ID]].drop_duplicates().shape[0],
        tuning_purchases[[USER_ID, COUPON_ID]].drop_duplicates().shape[0],
        valid_purchases[[USER_ID, COUPON_ID]].drop_duplicates().shape[0],
    ],
})
display(split_summary)
assert len(train_purchases) + len(tuning_purchases) + len(valid_purchases) == len(positive_events)


## 6. Negative Sampling

**做什么：** 每个训练正事件采样 `NEGATIVE_SAMPLE_RATIO` 个负券：同类别且价格/折扣最相近优先，其次相近价格段，最后随机补足。  
**为什么：** 完全随机负样本过于简单，不能训练出有辨别力的排序器。  
**输入：** 训练正事件、训练期可见优惠券、用户完整真实购买集合。  
**输出：** 带 `negative_source` 和参考正券的困难负样本。


In [ ]:
def make_sampling_index(coupon_pool):
    """Create reusable category and price-band lookup tables for hard-negative sampling."""
    meta = coupon_pool.drop_duplicates(COUPON_ID).set_index(COUPON_ID, drop=False).copy()
    meta.index.name = "_coupon_lookup_id"
    meta["GENRE_NAME"] = meta["GENRE_NAME"].fillna("__MISSING_GENRE__")
    log_price = meta["log_discount_price"].fillna(meta["log_discount_price"].median()).fillna(0)
    meta["price_band"] = pd.cut(log_price, bins=[-np.inf, 5.5, 6.5, 7.5, 8.5, np.inf], labels=False).fillna(-1).astype(int)
    genre_index = meta.groupby("GENRE_NAME", observed=True)[COUPON_ID].agg(list).to_dict()
    band_index = meta.groupby("price_band", observed=True)[COUPON_ID].agg(list).to_dict()
    return meta, genre_index, band_index


def sample_hard_negatives(positive_rows, coupon_pool, purchased_sets, n_negatives=5, seed=42):
    """Sample event-time-available L1/L2/L3 negatives without relabeling purchased coupons."""
    rng = np.random.default_rng(seed)
    meta, genre_index, band_index = make_sampling_index(coupon_pool)
    all_ids = meta[COUPON_ID].to_numpy(dtype=object)
    rows = []
    used_by_user = defaultdict(set)

    def available_frame(candidates, pos, blocked):
        """Filter candidate IDs by blocked set and availability at the positive event time."""
        candidate_index = pd.Index(np.asarray(candidates, dtype=object))
        eligible = candidate_index[~candidate_index.isin(blocked)]
        if len(eligible) == 0:
            return meta.iloc[0:0]
        cand = meta.loc[eligible]
        event_time = pd.Timestamp(pos["I_DATE"])
        available = (
            (cand["DISPFROM"].isna() | (cand["DISPFROM"] <= event_time)) &
            (cand["DISPEND"].isna() | (cand["DISPEND"] >= event_time))
        )
        return cand.loc[available]

    def nearest(candidates, pos, blocked, take):
        """Return approximate nearest coupons after a reproducible random pre-cap."""
        if take <= 0:
            return []
        cand = available_frame(candidates, pos, blocked)
        if len(cand) > HARD_POOL_LIMIT:
            sampled_ids = rng.choice(cand.index.to_numpy(), HARD_POOL_LIMIT, replace=False)
            cand = cand.loc[sampled_ids]
        if cand.empty:
            return []
        price = cand["log_discount_price"].fillna(0).to_numpy(dtype=float)
        rate = cand["PRICE_RATE"].fillna(0).to_numpy(dtype=float)
        period = cand["VALIDPERIOD"].fillna(0).to_numpy(dtype=float)
        distance = (
            np.abs(price - float(pos.get("log_discount_price", 0) or 0)) +
            np.abs(rate - float(pos.get("PRICE_RATE", 0) or 0)) / 100.0 +
            0.25 * np.abs(period - float(pos.get("VALIDPERIOD", 0) or 0)) / 365.0
        )
        take = min(take, len(cand))
        chosen = np.argpartition(distance, take - 1)[:take]
        chosen = chosen[np.argsort(distance[chosen])]
        return cand.index.to_numpy()[chosen].tolist()

    def random_available(candidates, pos, blocked, take):
        """Draw random fallback coupons only from event-time-available candidates."""
        if take <= 0:
            return []
        cand = available_frame(candidates, pos, blocked)
        if cand.empty:
            return []
        return rng.choice(cand.index.to_numpy(), min(take, len(cand)), replace=False).tolist()

    positives_with_meta = positive_rows.merge(
        meta[[COUPON_ID, "GENRE_NAME", "PRICE_RATE", "VALIDPERIOD", "log_discount_price", "price_band"]],
        on=COUPON_ID, how="left", validate="many_to_one",
    )
    for row in positives_with_meta.itertuples(index=False):
        pos = row._asdict()
        user_id, positive_coupon = str(pos[USER_ID]), str(pos[COUPON_ID])
        purchased = set(map(str, purchased_sets.get(user_id, set())))
        blocked = purchased | used_by_user[user_id]
        selected = []

        l1 = nearest(genre_index.get(pos.get("GENRE_NAME"), []), pos, blocked, n_negatives)
        selected.extend((cid, "L1_same_genre_similar") for cid in l1)
        blocked.update(l1)

        need = n_negatives - len(selected)
        l2 = nearest(band_index.get(pos.get("price_band", -1), []), pos, blocked, need)
        selected.extend((cid, "L2_similar_price_band") for cid in l2)
        blocked.update(l2)

        need = n_negatives - len(selected)
        l3 = random_available(all_ids, pos, blocked, need)
        selected.extend((cid, "L3_random_fallback") for cid in l3)

        for coupon_id, source in selected:
            rows.append((user_id, str(coupon_id), 0, pos["positive_event_id"], positive_coupon, source))
            used_by_user[user_id].add(str(coupon_id))

    columns = [USER_ID, COUPON_ID, "label", "positive_event_id", "reference_coupon_id", "negative_source"]
    return pd.DataFrame(rows, columns=columns)


train_coupon_pool = coupons_train[coupons_train["DISPFROM"].isna() | (coupons_train["DISPFROM"] < EARLY_STOP_START)].copy()
train_negatives = sample_hard_negatives(
    train_purchases, train_coupon_pool, all_purchased_sets,
    n_negatives=NEGATIVE_SAMPLE_RATIO, seed=SEED,
)
train_positives = train_purchases[[USER_ID, COUPON_ID, "label", "positive_event_id"]].copy()
train_positives["reference_coupon_id"] = train_positives[COUPON_ID]
train_positives["negative_source"] = "positive"
train_pairs = pd.concat([train_positives, train_negatives], ignore_index=True)
train_pairs = train_pairs.sort_values([USER_ID, "label"], ascending=[True, False], kind="stable").reset_index(drop=True)

negative_conflicts = sum(
    coupon_id in all_purchased_sets.get(user_id, set())
    for user_id, coupon_id in train_negatives[[USER_ID, COUPON_ID]].itertuples(index=False, name=None)
)
print("Negative source distribution:")
display(train_negatives["negative_source"].value_counts(dropna=False).to_frame("rows"))
print("Purchased coupons mislabeled as negative:", negative_conflicts)
assert negative_conflicts == 0


## 7. Feature Engineering

**做什么：** 分别构造用户、优惠券、历史行为、用户-优惠券交叉特征。  
**为什么：** 特征分层便于解释、复用和逐类排查；历史统计只使用验证切点之前的数据。  
**输入：** 用户表、券表、训练期购买与浏览。  
**输出：** 可用于三种排序模型的统一原始特征表。


In [ ]:
def build_history_features(history_purchases, history_visits, coupon_features):
    """Aggregate histories that are already cut off strictly before prediction time."""
    coupon_cols = [COUPON_ID, "GENRE_NAME", "DISCOUNT_PRICE", "PRICE_RATE", "ken_name"]
    hp = history_purchases.merge(coupon_features[coupon_cols], on=COUPON_ID, how="left")
    hv = history_visits.merge(coupon_features[[COUPON_ID, "GENRE_NAME"]], on=COUPON_ID, how="left")

    user_hist = hp.groupby(USER_ID, observed=True).agg(
        user_purchase_count=(COUPON_ID, "size"),
        user_unique_coupon_count=(COUPON_ID, "nunique"),
        user_unique_genre_count=("GENRE_NAME", "nunique"),
        user_avg_purchase_price=("DISCOUNT_PRICE", "mean"),
        user_median_purchase_price=("DISCOUNT_PRICE", "median"),
        user_avg_discount_rate=("PRICE_RATE", "mean"),
        _purchase_price_sum=("DISCOUNT_PRICE", "sum"),
        _discount_rate_sum=("PRICE_RATE", "sum"),
        _first_purchase=("I_DATE", "min"),
        _last_purchase=("I_DATE", "max"),
    ).reset_index()
    user_hist["user_active_days"] = (user_hist["_last_purchase"] - user_hist["_first_purchase"]).dt.days.clip(lower=0)

    if USE_VISIT_FEATURES:
        view_hist = hv.groupby(USER_ID, observed=True).agg(
            user_view_count=(COUPON_ID, "size"),
            user_unique_view_coupon_count=(COUPON_ID, "nunique"),
        ).reset_index()
        user_hist = user_hist.merge(view_hist, on=USER_ID, how="outer")
    else:
        user_hist["user_view_count"] = 0
        user_hist["user_unique_view_coupon_count"] = 0

    for col in ["user_purchase_count", "user_view_count"]:
        user_hist[col] = pd.to_numeric(user_hist[col], errors="coerce").fillna(0)
    user_hist["view_to_purchase_ratio"] = user_hist["user_view_count"] / user_hist["user_purchase_count"].replace(0, np.nan)
    user_hist["view_to_purchase_ratio"] = user_hist["view_to_purchase_ratio"].replace([np.inf, -np.inf], np.nan).fillna(0)

    user_genre = hp.groupby([USER_ID, "GENRE_NAME"], observed=True).size().rename("user_genre_purchase_count").reset_index()
    user_pref = hp.groupby([USER_ID, "ken_name"], observed=True).size().rename("user_pref_coupon_count").reset_index()
    user_coupon = hp.groupby([USER_ID, COUPON_ID], observed=True).size().rename("_user_coupon_purchase_count").reset_index()
    genre_view = hv.groupby([USER_ID, "GENRE_NAME"], observed=True).size().rename("user_genre_view_count").reset_index()
    return {"user": user_hist, "genre": user_genre, "pref": user_pref, "user_coupon": user_coupon, "genre_view": genre_view}


def build_user_features(user_frame, history):
    """Combine static user attributes and cutoff-safe historical aggregates."""
    base = user_frame[[USER_ID, "SEX_ID", "AGE", "PREF_NAME", "age_bucket"]].copy()
    base = base.rename(columns={"PREF_NAME": "USER_PREF_NAME"})
    return base.merge(history["user"], on=USER_ID, how="left", validate="one_to_one")


def build_coupon_features(coupon_frame):
    """Select consistent model-facing coupon attributes from train or test coupons."""
    desired = [
        COUPON_ID, "GENRE_NAME", "CAPSULE_TEXT", "PRICE_RATE", "CATALOG_PRICE", "DISCOUNT_PRICE",
        "DISPPERIOD", "VALIDPERIOD", "large_area_name", "ken_name", "small_area_name", "DISPFROM",
        "discount_amount", "discount_ratio", "log_catalog_price", "log_discount_price", "display_days",
        "disp_year", "disp_month", "disp_weekday", "disp_day",
    ]
    available = [col for col in desired if col in coupon_frame.columns]
    out = coupon_frame[available].copy()
    return out.rename(columns={"ken_name": "COUPON_PREF_NAME"})


def _strict_asof(query, state, by, query_time="_feature_event_time", state_time="_state_time"):
    """Attach the latest state strictly before query_time, compatible with pandas 1.0+."""
    by = list(by)
    left = query.copy()
    left["_original_order"] = np.arange(len(left), dtype=np.int64)
    for col in by:
        left[col] = left[col].fillna("__MISSING__").astype(str)
    right = state.copy()
    for col in by:
        right[col] = right[col].fillna("__MISSING__").astype(str)
    # merge_asof requires the time key to be globally sorted; group keys break ties.
    left = left.sort_values([query_time] + by, kind="stable")
    right = right.sort_values([state_time] + by, kind="stable")
    merged = pd.merge_asof(
        left, right, left_on=query_time, right_on=state_time, by=by,
        direction="backward", allow_exact_matches=False,
    )
    return merged.sort_values("_original_order", kind="stable").drop(columns="_original_order").reset_index(drop=True)


def _cumulative_count_state(events, keys, value_name):
    """Create cumulative event counts at each timestamp for an as-of join."""
    keys = list(keys)
    grouped = events.groupby(keys + ["I_DATE"], observed=True).size().rename("_increment").reset_index()
    grouped = grouped.sort_values(keys + ["I_DATE"], kind="stable")
    grouped[value_name] = grouped.groupby(keys, observed=True)["_increment"].cumsum()
    return grouped.drop(columns="_increment").rename(columns={"I_DATE": "_state_time"})


def build_event_time_training_features(pairs, user_frame, coupon_features, purchase_events, visit_events):
    """Build each training query from history strictly before its positive event.

    Every positive and negative attached to the same positive_event_id receives the
    same user-history snapshot. Candidate-specific genre/prefecture counts are then
    read from that same snapshot, eliminating the old positive-only LOO signal.
    """
    event_lookup = purchase_events[["positive_event_id", USER_ID, "I_DATE"]].drop_duplicates("positive_event_id")
    event_lookup = event_lookup.rename(columns={"I_DATE": "_feature_event_time"})
    static_users = user_frame[[USER_ID, "SEX_ID", "AGE", "PREF_NAME", "age_bucket"]].copy()
    static_users = static_users.rename(columns={"PREF_NAME": "USER_PREF_NAME"})

    frame = pairs.merge(event_lookup, on=["positive_event_id", USER_ID], how="left", validate="many_to_one")
    if frame["_feature_event_time"].isna().any():
        raise AssertionError("Some sampled rows are missing their reference event timestamp.")
    frame = frame.merge(static_users, on=USER_ID, how="left", validate="many_to_one")
    frame = frame.merge(coupon_features, on=COUPON_ID, how="left", validate="many_to_one")
    frame["GENRE_NAME"] = frame["GENRE_NAME"].fillna("__MISSING_GENRE__").astype(str)
    frame["COUPON_PREF_NAME"] = frame["COUPON_PREF_NAME"].fillna("__MISSING_PREF__").astype(str)

    history_meta = coupon_features[[COUPON_ID, "GENRE_NAME", "COUPON_PREF_NAME", "DISCOUNT_PRICE", "PRICE_RATE"]].copy()
    hp = purchase_events[[USER_ID, COUPON_ID, "I_DATE"]].merge(history_meta, on=COUPON_ID, how="left")
    hp["GENRE_NAME"] = hp["GENRE_NAME"].fillna("__MISSING_GENRE__").astype(str)
    hp["COUPON_PREF_NAME"] = hp["COUPON_PREF_NAME"].fillna("__MISSING_PREF__").astype(str)

    user_state = hp.groupby([USER_ID, "I_DATE"], observed=True).agg(
        _purchase_increment=(COUPON_ID, "size"),
        _price_increment=("DISCOUNT_PRICE", "sum"),
        _rate_increment=("PRICE_RATE", "sum"),
    ).reset_index().sort_values([USER_ID, "I_DATE"], kind="stable")
    user_group = user_state.groupby(USER_ID, observed=True)
    user_state["user_purchase_count"] = user_group["_purchase_increment"].cumsum()
    user_state["_purchase_price_sum"] = user_group["_price_increment"].cumsum()
    user_state["_discount_rate_sum"] = user_group["_rate_increment"].cumsum()
    user_state["_first_purchase"] = user_group["I_DATE"].transform("min")
    user_state["_history_last_purchase_time"] = user_state["I_DATE"]
    user_state = user_state.drop(columns=["_purchase_increment", "_price_increment", "_rate_increment"])
    user_state = user_state.rename(columns={"I_DATE": "_state_time"})

    event_query = frame[["positive_event_id", USER_ID, "_feature_event_time"]].drop_duplicates("positive_event_id")
    event_features = _strict_asof(event_query, user_state, [USER_ID])

    if USE_VISIT_FEATURES:
        visit_user_state = _cumulative_count_state(visit_events, [USER_ID], "user_view_count")
        event_features = _strict_asof(event_features.drop(columns=["_state_time"], errors="ignore"), visit_user_state, [USER_ID])
    else:
        event_features["user_view_count"] = 0
    event_features = event_features.drop(columns=["_state_time"], errors="ignore")
    event_features["user_purchase_count"] = event_features["user_purchase_count"].fillna(0)
    event_features["user_view_count"] = event_features["user_view_count"].fillna(0)
    event_features["user_avg_purchase_price"] = event_features["_purchase_price_sum"] / event_features["user_purchase_count"].replace(0, np.nan)
    event_features["user_avg_discount_rate"] = event_features["_discount_rate_sum"] / event_features["user_purchase_count"].replace(0, np.nan)
    event_features["user_active_days"] = (
        event_features["_feature_event_time"] - event_features["_first_purchase"]
    ).dt.days.clip(lower=0)
    event_features["view_to_purchase_ratio"] = event_features["user_view_count"] / event_features["user_purchase_count"].replace(0, np.nan)
    frame = frame.merge(
        event_features.drop(columns="_feature_event_time"),
        on=["positive_event_id", USER_ID], how="left", validate="many_to_one",
    )

    genre_state = _cumulative_count_state(hp, [USER_ID, "GENRE_NAME"], "user_genre_purchase_count")
    frame = _strict_asof(frame, genre_state, [USER_ID, "GENRE_NAME"])
    frame = frame.drop(columns="_state_time", errors="ignore")

    pref_state = _cumulative_count_state(hp, [USER_ID, "COUPON_PREF_NAME"], "user_pref_coupon_count")
    frame = _strict_asof(frame, pref_state, [USER_ID, "COUPON_PREF_NAME"])
    frame = frame.drop(columns="_state_time", errors="ignore")

    if USE_VISIT_FEATURES:
        hv = visit_events[[USER_ID, COUPON_ID, "I_DATE"]].merge(
            coupon_features[[COUPON_ID, "GENRE_NAME"]], on=COUPON_ID, how="left"
        )
        hv["GENRE_NAME"] = hv["GENRE_NAME"].fillna("__MISSING_GENRE__").astype(str)
        genre_view_state = _cumulative_count_state(hv, [USER_ID, "GENRE_NAME"], "user_genre_view_count")
        frame = _strict_asof(frame, genre_view_state, [USER_ID, "GENRE_NAME"])
        frame = frame.drop(columns="_state_time", errors="ignore")
    else:
        frame["user_genre_view_count"] = 0

    for col in ["user_genre_purchase_count", "user_pref_coupon_count", "user_genre_view_count"]:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(0)
    frame["same_prefecture"] = (
        frame["USER_PREF_NAME"].fillna("") == frame["COUPON_PREF_NAME"].fillna("__NONE__")
    ).astype("int8")
    frame["user_genre_purchase_ratio"] = frame["user_genre_purchase_count"] / frame["user_purchase_count"].replace(0, np.nan)
    frame["user_genre_view_ratio"] = frame["user_genre_view_count"] / frame["user_view_count"].replace(0, np.nan)
    frame["price_gap"] = (frame["DISCOUNT_PRICE"] - frame["user_avg_purchase_price"]).abs()
    frame["discount_rate_gap"] = (frame["PRICE_RATE"] - frame["user_avg_discount_rate"]).abs()
    return frame.replace([np.inf, -np.inf], np.nan)


def build_user_coupon_features(pairs, user_features, coupon_features, history):
    """Build inference/evaluation features from a history already cut off in time."""
    frame = pairs.merge(user_features, on=USER_ID, how="left", validate="many_to_one")
    frame = frame.merge(coupon_features, on=COUPON_ID, how="left", validate="many_to_one")
    frame = frame.merge(history["genre"], on=[USER_ID, "GENRE_NAME"], how="left")
    frame = frame.merge(history["pref"], left_on=[USER_ID, "COUPON_PREF_NAME"], right_on=[USER_ID, "ken_name"], how="left")
    frame = frame.drop(columns=["ken_name"], errors="ignore")
    frame = frame.merge(history["user_coupon"], on=[USER_ID, COUPON_ID], how="left")
    frame = frame.merge(history["genre_view"], on=[USER_ID, "GENRE_NAME"], how="left")

    numeric_defaults = [
        "user_purchase_count", "user_unique_coupon_count", "user_unique_genre_count", "user_avg_purchase_price",
        "user_median_purchase_price", "user_avg_discount_rate", "user_active_days", "user_view_count",
        "user_unique_view_coupon_count", "view_to_purchase_ratio", "user_genre_purchase_count",
        "user_pref_coupon_count", "_user_coupon_purchase_count", "user_genre_view_count",
        "_purchase_price_sum", "_discount_rate_sum",
    ]
    for col in numeric_defaults:
        if col not in frame:
            frame[col] = 0
        frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(0)

    frame["same_prefecture"] = (frame["USER_PREF_NAME"].fillna("") == frame["COUPON_PREF_NAME"].fillna("__NONE__")).astype("int8")
    frame["user_genre_purchase_ratio"] = frame["user_genre_purchase_count"] / frame["user_purchase_count"].replace(0, np.nan)
    frame["user_genre_view_ratio"] = frame["user_genre_view_count"] / frame["user_view_count"].replace(0, np.nan)
    frame["price_gap"] = (frame["DISCOUNT_PRICE"] - frame["user_avg_purchase_price"]).abs()
    frame["discount_rate_gap"] = (frame["PRICE_RATE"] - frame["user_avg_discount_rate"]).abs()
    return frame.replace([np.inf, -np.inf], np.nan)


history_visits_train = visits[visits["I_DATE"] < EARLY_STOP_START].copy()
history_train = build_history_features(train_purchases, history_visits_train, coupons_train)
user_features_train = build_user_features(users, history_train)
coupon_features_train = build_coupon_features(coupons_train)
train_raw = build_event_time_training_features(
    train_pairs, users, coupon_features_train, train_purchases, history_visits_train,
)
print("Training raw feature shape:", train_raw.shape)
display(train_raw.head(3))


## 8. Dataset Construction

**做什么：** 构造早停候选池，并在最终留出期对全部可用优惠券进行排序。  
**为什么：** 训练负样本用于学习；早停候选用于确定迭代轮数；最终 MAP@10 必须在未参与早停的全量候选池上报告。  
**输入：** 两个未来时间段的真实购买、当期可用券和各自预测时点之前的历史。  
**输出：** `tuning_raw` 与独立的全量候选 `valid_raw`。


In [ ]:
def build_validation_candidates(valid_truth, coupon_pool, history_purchases, history_visits, max_per_user=500, seed=42):
    """Optional retrieval diagnostic; it is not used for ranker early stopping."""
    rng = np.random.default_rng(seed)
    pool_ids = set(coupon_pool[COUPON_ID].astype(str))
    popularity = history_purchases[COUPON_ID].value_counts()
    popular_ids = [cid for cid in popularity.index.astype(str) if cid in pool_ids][:POPULAR_CANDIDATES]
    truth_sets = valid_truth.groupby(USER_ID, observed=True)[COUPON_ID].agg(lambda s: set(s.astype(str))).to_dict()
    rows, hits, total = [], 0, 0
    for user_id, truth in truth_sets.items():
        selected = set(popular_ids)
        eligible = np.array(list(pool_ids - selected), dtype=object)
        if len(selected) < max_per_user and len(eligible):
            selected.update(rng.choice(eligible, min(max_per_user - len(selected), len(eligible)), replace=False).tolist())
        hits += len(selected & truth)
        total += len(truth)
        rows.extend((str(user_id), str(cid), int(cid in truth)) for cid in selected)
    return pd.DataFrame(rows, columns=[USER_ID, COUPON_ID, "label"]), hits / max(1, total)


def build_full_candidate_pairs(truth, coupon_pool):
    """Create an all-available-coupon pool for every evaluation user."""
    users_eval = truth[USER_ID].drop_duplicates().astype(str).to_numpy()
    coupon_ids = coupon_pool[COUPON_ID].drop_duplicates().astype(str).to_numpy()
    pairs = pd.DataFrame({
        USER_ID: np.repeat(users_eval, len(coupon_ids)),
        COUPON_ID: np.tile(coupon_ids, len(users_eval)),
    })
    truth_keys = set(zip(truth[USER_ID].astype(str), truth[COUPON_ID].astype(str)))
    pairs["label"] = np.fromiter(
        ((user_id, coupon_id) in truth_keys for user_id, coupon_id in pairs[[USER_ID, COUPON_ID]].itertuples(index=False, name=None)),
        dtype=np.int8, count=len(pairs),
    )
    return pairs


# Early stopping and untouched holdout use the same full-pool protocol.
tuning_truth = tuning_purchases[[USER_ID, COUPON_ID]].drop_duplicates().copy()
tuning_user_ids = tuning_truth[USER_ID].drop_duplicates().to_numpy()
if MAX_TUNING_USERS and len(tuning_user_ids) > MAX_TUNING_USERS:
    tuning_user_ids = np.sort(np.random.default_rng(SEED).choice(tuning_user_ids, MAX_TUNING_USERS, replace=False))
    tuning_truth = tuning_truth[tuning_truth[USER_ID].isin(tuning_user_ids)].copy()
tuning_coupon_pool = coupons_train[
    (coupons_train["DISPFROM"].isna() | (coupons_train["DISPFROM"] <= VALID_START)) &
    (coupons_train["DISPEND"].isna() | (coupons_train["DISPEND"] >= EARLY_STOP_START))
].copy()
tuning_coupon_pool = pd.concat([
    tuning_coupon_pool, coupons_train[coupons_train[COUPON_ID].isin(set(tuning_truth[COUPON_ID]))]
]).drop_duplicates(COUPON_ID)
tuning_pairs = build_full_candidate_pairs(tuning_truth, tuning_coupon_pool)
tuning_raw = build_user_coupon_features(
    tuning_pairs, user_features_train, coupon_features_train, history_train,
).sort_values(USER_ID, kind="stable").reset_index(drop=True)

# Untouched holdout uses histories available strictly before VALID_START.
valid_truth = valid_purchases[[USER_ID, COUPON_ID]].drop_duplicates().copy()
valid_user_ids = valid_truth[USER_ID].drop_duplicates().to_numpy()
if MAX_VALID_USERS and len(valid_user_ids) > MAX_VALID_USERS:
    valid_user_ids = np.sort(np.random.default_rng(SEED + 1).choice(valid_user_ids, MAX_VALID_USERS, replace=False))
    valid_truth = valid_truth[valid_truth[USER_ID].isin(valid_user_ids)].copy()
pre_valid_purchases = pd.concat([train_purchases, tuning_purchases], ignore_index=True)
history_visits_valid = visits[visits["I_DATE"] < VALID_START].copy()
history_valid = build_history_features(pre_valid_purchases, history_visits_valid, coupons_train)
user_features_valid = build_user_features(users, history_valid)
valid_coupon_pool = coupons_train[
    (coupons_train["DISPFROM"].isna() | (coupons_train["DISPFROM"] <= VALID_END)) &
    (coupons_train["DISPEND"].isna() | (coupons_train["DISPEND"] >= VALID_START))
].copy()
valid_coupon_pool = pd.concat([
    valid_coupon_pool, coupons_train[coupons_train[COUPON_ID].isin(set(valid_truth[COUPON_ID]))]
]).drop_duplicates(COUPON_ID)
valid_pairs = build_full_candidate_pairs(valid_truth, valid_coupon_pool)
valid_raw = build_user_coupon_features(
    valid_pairs, user_features_valid, coupon_features_train, history_valid,
).sort_values(USER_ID, kind="stable").reset_index(drop=True)
train_raw = train_raw.sort_values(USER_ID, kind="stable").reset_index(drop=True)

print("Train dataset:", train_raw.shape)
print("Early-stop full pool:", tuning_raw.shape, "active coupons:", tuning_coupon_pool[COUPON_ID].nunique())
print("Untouched holdout full pool:", valid_raw.shape, "active coupons:", valid_coupon_pool[COUPON_ID].nunique())
print("Official test coupon count:", coupons_test[COUPON_ID].nunique())
display(tuning_raw.groupby(USER_ID).size().describe().to_frame("tuning_candidate_count"))
display(valid_raw.groupby(USER_ID).size().describe().to_frame("holdout_candidate_count"))


### Leakage Check

检查时间分区、事件 ID、标签来源、地区标签关系、测试券隔离和负样本冲突。地区可以具有预测力，但不能确定性地产生标签。


In [ ]:
region_label_counts = pd.crosstab(train_raw["same_prefecture"], train_raw["label"])
region_has_both_labels = (
    {0, 1}.issubset(set(region_label_counts.columns)) and
    (region_label_counts[[0, 1]] > 0).all(axis=1).all()
)
future_event_ids = set(tuning_purchases["positive_event_id"]) | set(valid_purchases["positive_event_id"])
train_event_ids = set(train_purchases["positive_event_id"])
labels_match_provenance = train_pairs["label"].eq(train_pairs["negative_source"].eq("positive").astype(np.int8)).all()
history_time_ok = (
    train_raw["_history_last_purchase_time"].isna() |
    (train_raw["_history_last_purchase_time"] < train_raw["_feature_event_time"])
).all()
shared_history_columns = [
    "user_purchase_count", "user_avg_purchase_price", "user_avg_discount_rate",
    "user_view_count", "user_active_days", "view_to_purchase_ratio",
]
event_history_is_shared = all(
    train_raw.groupby("positive_event_id", observed=True)[col].nunique(dropna=False).max() <= 1
    for col in shared_history_columns
)

leakage_checks = {
    "future purchase events excluded from training history": train_event_ids.isdisjoint(future_event_ids),
    "training history is strictly earlier than each event": bool(history_time_ok),
    "positive and negatives share one event-history snapshot": bool(event_history_is_shared),
    "history visits strictly before early-stop cutoff": history_visits_train.empty or history_visits_train["I_DATE"].max() < EARLY_STOP_START,
    "three time partitions strictly ordered": (
        train_purchases["I_DATE"].max() < tuning_purchases["I_DATE"].min() and
        tuning_purchases["I_DATE"].max() < valid_purchases["I_DATE"].min()
    ),
    "labels exactly match purchase/negative provenance": labels_match_provenance,
    "region feature cannot deterministically derive label": region_has_both_labels,
    "test coupons absent from supervised labels": not set(coupons_test[COUPON_ID]).intersection(set(train_pairs[COUPON_ID])),
    "no purchased coupon sampled negative": negative_conflicts == 0,
    "every training positive event retained": len(train_positives) == len(train_purchases),
}
display(pd.Series(leakage_checks, name="passed").to_frame())
display(region_label_counts)
assert all(leakage_checks.values()), "At least one leakage check failed."


def query_invariant_features(frame):
    """Report columns constant within every user query; tree interactions may still use them."""
    ignored = NON_FEATURE_COLUMNS if "NON_FEATURE_COLUMNS" in globals() else {USER_ID, COUPON_ID, "label"}
    candidates = [col for col in frame.columns if col not in ignored]
    return [
        col for col in candidates
        if frame.groupby(USER_ID, observed=True)[col].nunique(dropna=False).max() <= 1
    ]


invariant_features = query_invariant_features(train_raw)
print("Query-invariant feature audit (review/remove or deliberately interact):")
print(invariant_features)


### Sanity Check

训练前检查标签比例、实体规模和困难负样本的可解释性。随机展示 3 个用户的正券及其对应困难负券。


In [ ]:
sanity = {
    "positive_samples": int(train_pairs["label"].sum()),
    "negative_samples": int((train_pairs["label"] == 0).sum()),
    "positive_ratio": float(train_pairs["label"].mean()),
    "unique_users": int(train_pairs[USER_ID].nunique()),
    "unique_coupons": int(train_pairs[COUPON_ID].nunique()),
    "avg_samples_per_user": float(train_pairs.groupby(USER_ID).size().mean()),
}
display(pd.Series(sanity, name="value").to_frame())

sample_users = np.random.default_rng(SEED).choice(train_pairs[USER_ID].unique(), size=min(3, train_pairs[USER_ID].nunique()), replace=False)
audit = train_pairs[train_pairs[USER_ID].isin(sample_users)].merge(
    coupons_train[[COUPON_ID, "GENRE_NAME", "DISCOUNT_PRICE", "PRICE_RATE"]], on=COUPON_ID, how="left"
)
audit = audit.sort_values([USER_ID, "positive_event_id", "label"], ascending=[True, True, False])
display(audit[[USER_ID, "positive_event_id", COUPON_ID, "reference_coupon_id", "GENRE_NAME", "DISCOUNT_PRICE", "PRICE_RATE", "label", "negative_source"]].groupby(USER_ID).head(12))


## 9. MAP@10 Evaluation

**做什么：** 实现 Ponpare 风格的 AP@10 / MAP@10，并提供统一模型评估函数。  
**为什么：** Accuracy、F1 或 AUC 不衡量每位用户 Top-10 的顺序质量。  
**输入：** 每个用户的真实券集合与预测分数。  
**输出：** MAP@10 和去重 Top-10 推荐。


In [ ]:
def apk(actual, predicted, k=10):
    """Average precision at k with duplicate predictions ignored."""
    actual_set = set(actual)
    if not actual_set:
        return 0.0
    score, hits = 0.0, 0
    unique_predicted = list(dict.fromkeys(predicted))[:k]
    for rank, item in enumerate(unique_predicted, start=1):
        if item in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)


def per_user_apk(actual_by_user, predicted_by_user, k=10):
    """Return AP@k indexed by user so paired uncertainty estimates remain possible."""
    return pd.Series({
        user: apk(actual, predicted_by_user.get(user, []), k)
        for user, actual in actual_by_user.items()
    }, name=f"AP@{k}", dtype=float)


def mapk(actual_by_user, predicted_by_user, k=10):
    return float(per_user_apk(actual_by_user, predicted_by_user, k).mean())


def rank_predictions(frame, scores, k=10):
    """Convert row scores into unique per-user ranked coupon lists."""
    ranked = frame[[USER_ID, COUPON_ID]].copy()
    ranked["score"] = np.asarray(scores)
    ranked = ranked.sort_values([USER_ID, "score"], ascending=[True, False], kind="stable")
    ranked = ranked.drop_duplicates([USER_ID, COUPON_ID]).groupby(USER_ID, observed=True, sort=False).head(k)
    return ranked


def evaluate_scores(frame, scores, truth, k=10):
    """Evaluate scores and retain the per-user AP vector for paired bootstrap."""
    ranked = rank_predictions(frame, scores, k)
    actual = truth.groupby(USER_ID, observed=True)[COUPON_ID].agg(lambda s: list(pd.unique(s.astype(str)))).to_dict()
    predicted = ranked.groupby(USER_ID, observed=True)[COUPON_ID].agg(list).to_dict()
    per_user = per_user_apk(actual, predicted, k)
    return float(per_user.mean()), ranked, per_user


def paired_bootstrap_delta(reference, challenger, repeats=2000, seed=42):
    """Paired user bootstrap CI for mean(reference AP - challenger AP)."""
    aligned = pd.concat([reference.rename("reference"), challenger.rename("challenger")], axis=1).dropna()
    delta = (aligned["reference"] - aligned["challenger"]).to_numpy(dtype=float)
    if len(delta) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = np.empty(repeats, dtype=float)
    for i in range(repeats):
        boot[i] = delta[rng.integers(0, len(delta), len(delta))].mean()
    low, high = np.percentile(boot, [2.5, 97.5])
    return float(delta.mean()), float(low), float(high)


assert abs(apk(["a", "b"], ["a", "x", "b"], 10) - ((1/1 + 2/3)/2)) < 1e-12
assert apk(["a"], ["x", "a"], 1) == 0.0
assert apk(["a"], ["a", "a"], 10) == 1.0
print("MAP@10 and paired-bootstrap helpers passed unit checks.")


## 10. LightGBM Ranker

**做什么：** 使用 `LGBMRanker(objective='lambdarank')` 建立主基线。  
**为什么：** LambdaRank 对分组排序稳定、训练速度快，适合作为 V1.1 基准。  
**输入：** 用户内连续的数值编码特征及 group size。  
**输出：** GPU 优先的 LightGBM 模型、实际设备、MAP@10、耗时和特征重要性。


In [ ]:
CATEGORICAL_FEATURES = [
    "SEX_ID", "USER_PREF_NAME", "age_bucket", "GENRE_NAME", "CAPSULE_TEXT",
    "large_area_name", "COUPON_PREF_NAME", "small_area_name",
]
NON_FEATURE_COLUMNS = {
    USER_ID, COUPON_ID, "label", "positive_event_id", "reference_coupon_id", "negative_source",
    "DISPFROM", "coupon_split", "_first_purchase", "_last_purchase", "_purchase_price_sum",
    "_discount_rate_sum", "_user_coupon_purchase_count", "_feature_event_time",
    "_history_last_purchase_time", "_state_time", "disp_year", "disp_month", "disp_day",
}


def fit_preprocessor(train_frame):
    """Fit train-only category maps and numeric fill values shared by tree rankers."""
    feature_columns = [c for c in train_frame.columns if c not in NON_FEATURE_COLUMNS]
    categorical = [c for c in CATEGORICAL_FEATURES if c in feature_columns]
    numeric = [c for c in feature_columns if c not in categorical]
    category_maps = {}
    for col in categorical:
        values = train_frame[col].fillna("__MISSING__").astype(str)
        category_maps[col] = {value: i + 1 for i, value in enumerate(sorted(values.unique()))}
    numeric_fill = {
        col: float(pd.to_numeric(train_frame[col], errors="coerce").median())
        if pd.to_numeric(train_frame[col], errors="coerce").notna().any() else 0.0
        for col in numeric
    }
    return {"features": feature_columns, "categorical": categorical, "numeric": numeric,
            "category_maps": category_maps, "numeric_fill": numeric_fill}


def transform_numeric(frame, preprocessor):
    """Apply compact ordinal category encoding without one-hot expansion."""
    out = pd.DataFrame(index=frame.index)
    for col in preprocessor["categorical"]:
        out[col] = frame[col].fillna("__MISSING__").astype(str).map(preprocessor["category_maps"][col]).fillna(0).astype("int32")
    for col in preprocessor["numeric"]:
        out[col] = pd.to_numeric(frame[col], errors="coerce").fillna(preprocessor["numeric_fill"][col]).astype("float32")
    return out[preprocessor["features"]]


def transform_catboost(frame, preprocessor):
    """Preserve raw strings for CatBoost while applying identical numeric imputation."""
    out = pd.DataFrame(index=frame.index)
    for col in preprocessor["categorical"]:
        out[col] = frame[col].fillna("__MISSING__").astype(str)
    for col in preprocessor["numeric"]:
        out[col] = pd.to_numeric(frame[col], errors="coerce").fillna(preprocessor["numeric_fill"][col]).astype("float32")
    return out[preprocessor["features"]]


def ranking_query_column(frame):
    """Use sampled purchase events for training and users for full-pool evaluation."""
    if "positive_event_id" in frame and frame["positive_event_id"].notna().all():
        return "positive_event_id"
    return USER_ID


def group_sizes(frame, query_col=None):
    """Return ranking group sizes after asserting that query rows are contiguous."""
    query_col = query_col or ranking_query_column(frame)
    query_values = frame[query_col].astype(str).to_numpy()
    if len(query_values) > 1:
        starts = np.r_[True, query_values[1:] != query_values[:-1]]
        seen_order = query_values[starts]
        if len(seen_order) != len(set(seen_order)):
            raise AssertionError(f"Ranking query rows are not contiguous: {query_col}")
    groups = frame.groupby(query_col, sort=False, observed=True).size().to_numpy()
    assert groups.sum() == len(frame)
    return groups


def extract_best_iteration(model_name, model, fallback):
    """Return a positive one-based tree/iteration count for uncontaminated full refitting."""
    if model_name == "LightGBM":
        value = getattr(model, "best_iteration_", None)
        return int(value) if value is not None and int(value) > 0 else int(fallback)
    if model_name == "XGBoost":
        value = getattr(model, "best_iteration", None)
        return int(value) + 1 if value is not None and int(value) >= 0 else int(fallback)
    if model_name == "CatBoost" and hasattr(model, "get_best_iteration"):
        value = model.get_best_iteration()
        return int(value) + 1 if value is not None and int(value) >= 0 else int(fallback)
    return int(fallback)


def model_gain_importance(artifact):
    """Return comparable gain-based importance where the library supports it."""
    model = artifact["model"]
    features = artifact["preprocessor"]["features"]
    if artifact["name"] == "LightGBM":
        return np.asarray(model.booster_.feature_importance(importance_type="gain"), dtype=float), "gain"
    if artifact["name"] == "XGBoost":
        scores = model.get_booster().get_score(importance_type="gain")
        values = [scores.get(name, scores.get(f"f{i}", 0.0)) for i, name in enumerate(features)]
        return np.asarray(values, dtype=float), "gain"
    return np.asarray(model.get_feature_importance(type="PredictionValuesChange"), dtype=float), "PredictionValuesChange"


def fit_ranker(model_name, train_frame, valid_frame=None, fixed_iterations=None):
    """Fit one ranker with GPU priority and a model-local CPU fallback."""
    train_frame = train_frame.copy()
    train_frame[USER_ID] = train_frame[USER_ID].astype(str)
    train_query_col = ranking_query_column(train_frame)
    train_frame = train_frame.sort_values([train_query_col, USER_ID], kind="stable").reset_index(drop=True)
    if valid_frame is not None:
        valid_frame = valid_frame.copy()
        valid_frame[USER_ID] = valid_frame[USER_ID].astype(str)
        valid_query_col = ranking_query_column(valid_frame)
        valid_frame = valid_frame.sort_values([valid_query_col, USER_ID], kind="stable").reset_index(drop=True)
    else:
        valid_query_col = None
    prep = fit_preprocessor(train_frame)
    g_train = group_sizes(train_frame, train_query_col)
    y_train = train_frame["label"].to_numpy(dtype=np.int8)
    prefer_gpu = bool(USE_GPU and GPU_AVAILABLE)
    actual_device = "CPU"
    requested_iterations = int(fixed_iterations or 500)
    start = time.perf_counter()

    def run_with_fallback(train_callable):
        nonlocal actual_device
        if prefer_gpu:
            try:
                model = train_callable(True)
                actual_device = "GPU"
                return model
            except Exception as exc:
                print(f"{model_name} GPU attempt failed ({type(exc).__name__}: {exc}). Retrying on CPU.")
                gc.collect()
        actual_device = "CPU"
        return train_callable(False)

    if model_name == "LightGBM":
        x_train = transform_numeric(train_frame, prep)
        x_valid = transform_numeric(valid_frame, prep) if valid_frame is not None else None
        # Pandas categorical dtype lets LightGBM auto-detect categories without duplicate declarations.
        for col in prep["categorical"]:
            x_train[col] = x_train[col].astype("category")
            if x_valid is not None:
                x_valid[col] = x_valid[col].astype("category")

        def train_lightgbm(use_gpu):
            params = dict(
                objective="lambdarank", metric="map", eval_at=[TOP_K], learning_rate=0.05,
                n_estimators=requested_iterations, num_leaves=31, max_depth=-1, min_child_samples=30,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                random_state=SEED, n_jobs=-1, verbosity=-1,
            )
            if use_gpu:
                params.update(device_type="gpu", gpu_device_id=GPU_DEVICE_ID)
            model = lgb.LGBMRanker(**params)
            fit_kwargs = {"group": g_train}
            if valid_frame is not None:
                fit_kwargs.update({
                    "eval_set": [(x_valid, valid_frame["label"].to_numpy())],
                    "eval_group": [group_sizes(valid_frame, valid_query_col)], "eval_at": [TOP_K],
                    "callbacks": [lgb.early_stopping(50, verbose=False)],
                })
            model.fit(x_train, y_train, **fit_kwargs)
            return model

        model = run_with_fallback(train_lightgbm)

    elif model_name == "XGBoost":
        x_train = transform_numeric(train_frame, prep)
        x_valid = transform_numeric(valid_frame, prep) if valid_frame is not None else None
        version_parts = tuple(int(part) for part in re.findall(r"\d+", xgb.__version__)[:2])

        def train_xgboost(use_gpu):
            params = dict(
                objective="rank:ndcg", eval_metric=f"map@{TOP_K}", learning_rate=0.05,
                n_estimators=requested_iterations, max_depth=7, min_child_weight=5, subsample=0.8,
                colsample_bytree=0.8, reg_lambda=1.0, random_state=SEED, n_jobs=-1,
            )
            if use_gpu:
                if version_parts >= (2, 0):
                    params.update(tree_method="hist", device="cuda")
                else:
                    params.update(tree_method="gpu_hist", gpu_id=GPU_DEVICE_ID)
            else:
                params["tree_method"] = "hist"
                if version_parts >= (2, 0):
                    params["device"] = "cpu"
            if valid_frame is not None and version_parts >= (1, 6):
                params["callbacks"] = [xgb.callback.EarlyStopping(rounds=50, save_best=True, maximize=True)]
            model = xgb.XGBRanker(**params)
            fit_signature = inspect.signature(model.fit).parameters
            fit_kwargs = {"verbose": False}
            if "qid" in fit_signature:
                fit_kwargs["qid"] = pd.factorize(train_frame[train_query_col], sort=False)[0]
            else:
                fit_kwargs["group"] = g_train
            if valid_frame is not None:
                fit_kwargs["eval_set"] = [(x_valid, valid_frame["label"].to_numpy())]
                if "eval_qid" in fit_signature:
                    fit_kwargs["eval_qid"] = [pd.factorize(valid_frame[valid_query_col], sort=False)[0]]
                else:
                    fit_kwargs["eval_group"] = [group_sizes(valid_frame, valid_query_col)]
                if version_parts < (1, 6):
                    fit_kwargs["early_stopping_rounds"] = 50
            model.fit(x_train, y_train, **fit_kwargs)
            return model

        model = run_with_fallback(train_xgboost)

    elif model_name == "CatBoost":
        x_train = transform_catboost(train_frame, prep)
        train_pool = catboost.Pool(
            x_train, y_train, group_id=train_frame[train_query_col].astype(str).to_numpy(),
            cat_features=prep["categorical"],
        )
        valid_pool = None
        if valid_frame is not None:
            valid_pool = catboost.Pool(
                transform_catboost(valid_frame, prep), valid_frame["label"].to_numpy(),
                group_id=valid_frame[valid_query_col].astype(str).to_numpy(), cat_features=prep["categorical"],
            )

        def train_catboost(use_gpu):
            params = dict(
                loss_function="YetiRank", eval_metric=f"MAP:top={TOP_K}", iterations=requested_iterations,
                learning_rate=0.05, depth=7, l2_leaf_reg=5, random_seed=SEED,
                verbose=100, allow_writing_files=False, task_type="GPU" if use_gpu else "CPU",
            )
            if use_gpu:
                params["devices"] = str(GPU_DEVICE_ID)
            ranker_class = getattr(catboost, "CatBoostRanker", None)
            if ranker_class is not None:
                model = ranker_class(**params)
            else:
                generic_class = getattr(catboost, "CatBoost", None)
                if generic_class is None:
                    raise AttributeError("The imported catboost module exposes neither CatBoostRanker nor CatBoost.")
                print("CatBoostRanker is unavailable; using CatBoost(params) with YetiRank.")
                model = generic_class(params=params)
            fit_kwargs = {}
            if valid_pool is not None:
                fit_kwargs.update({"eval_set": valid_pool, "use_best_model": True, "early_stopping_rounds": 50})
            model.fit(train_pool, **fit_kwargs)
            return model

        model = run_with_fallback(train_catboost)
    else:
        raise ValueError(model_name)

    best_iteration = extract_best_iteration(model_name, model, requested_iterations)
    print(f"{model_name} trained on: {actual_device}; selected iterations: {best_iteration}")
    return {
        "name": model_name, "model": model, "preprocessor": prep,
        "train_time": time.perf_counter() - start, "device": actual_device,
        "best_iteration": best_iteration,
    }


def predict_ranker(artifact, frame):
    prep = artifact["preprocessor"]
    if artifact["name"] == "CatBoost":
        x = transform_catboost(frame, prep)
    else:
        x = transform_numeric(frame, prep)
        if artifact["name"] == "LightGBM":
            for col in prep["categorical"]:
                x[col] = x[col].astype("category")
    return np.asarray(artifact["model"].predict(x)).reshape(-1)


model_artifacts = {}
experiment_rows = []

if TRAIN_LIGHTGBM:
    artifact = fit_ranker("LightGBM", train_raw, tuning_raw)
    pred_start = time.perf_counter()
    scores = predict_ranker(artifact, valid_raw)
    predict_time = time.perf_counter() - pred_start
    map10, _, per_user_ap = evaluate_scores(valid_raw, scores, valid_truth, TOP_K)
    artifact.update({"map10": map10, "predict_time": predict_time, "per_user_ap": per_user_ap})
    model_artifacts["LightGBM"] = artifact
    experiment_rows.append({"model": "LightGBM", "kind": "ranker", "device": artifact["device"], "MAP@10": map10,
                            "best_iteration": artifact["best_iteration"], "train_time": artifact["train_time"], "predict_time": predict_time})
    print(f"LightGBM MAP@10={map10:.6f}")
    values, importance_type = model_gain_importance(artifact)
    lgb_importance = pd.DataFrame({"feature": artifact["preprocessor"]["features"], "importance": values}).sort_values("importance", ascending=False)
    display(lgb_importance.head(20))
    lgb_importance.head(20).sort_values("importance").plot.barh(x="feature", y="importance", legend=False, figsize=(8, 6))
    plt.title(f"LightGBM Top 20 Feature Importance ({importance_type})")
    plt.tight_layout()
    plt.show()


## 11. XGBoost Ranker

**做什么：** 在相同数据上训练 `XGBRanker(rank:ndcg)`。  
**为什么：** 观察另一种高效梯度提升实现对困难负样本的排序能力。  
**输入：** 与 LightGBM 完全相同的编码特征、标签和 group。  
**输出：** XGBoost MAP@10 与耗时。


In [ ]:
if TRAIN_XGBOOST:
    artifact = fit_ranker("XGBoost", train_raw, tuning_raw)
    pred_start = time.perf_counter()
    scores = predict_ranker(artifact, valid_raw)
    predict_time = time.perf_counter() - pred_start
    map10, _, per_user_ap = evaluate_scores(valid_raw, scores, valid_truth, TOP_K)
    artifact.update({"map10": map10, "predict_time": predict_time, "per_user_ap": per_user_ap})
    model_artifacts["XGBoost"] = artifact
    experiment_rows.append({"model": "XGBoost", "kind": "ranker", "device": artifact["device"], "MAP@10": map10,
                            "best_iteration": artifact["best_iteration"], "train_time": artifact["train_time"], "predict_time": predict_time})
    print(f"XGBoost MAP@10={map10:.6f}")


## 12. CatBoost Ranker

**做什么：** 使用原生类别特征训练 `CatBoostRanker(YetiRank)`。  
**为什么：** CatBoost 无需 one-hot，可以公平比较其类别处理能力。  
**输入：** 同一原始特征、标签和连续 group id。  
**输出：** CatBoost MAP@10 与耗时。


In [ ]:
if TRAIN_CATBOOST:
    artifact = fit_ranker("CatBoost", train_raw, tuning_raw)
    pred_start = time.perf_counter()
    scores = predict_ranker(artifact, valid_raw)
    predict_time = time.perf_counter() - pred_start
    map10, _, per_user_ap = evaluate_scores(valid_raw, scores, valid_truth, TOP_K)
    artifact.update({"map10": map10, "predict_time": predict_time, "per_user_ap": per_user_ap})
    model_artifacts["CatBoost"] = artifact
    experiment_rows.append({"model": "CatBoost", "kind": "ranker", "device": artifact["device"], "MAP@10": map10,
                            "best_iteration": artifact["best_iteration"], "train_time": artifact["train_time"], "predict_time": predict_time})
    print(f"CatBoost MAP@10={map10:.6f}")


## 13. Model Comparison

**做什么：** 汇总 MAP@10、训练耗时、预测耗时并绘图。  
**为什么：** 早停只使用调优时间段，三模型最终比较统一使用未参与早停的全量候选留出集。  
**输入：** 三个模型在独立留出集上的实验日志。  
**输出：** 排序后的对比表、最佳模型和 Top-20 特征重要性。


In [ ]:
if not experiment_rows:
    raise RuntimeError("Enable at least one ranker.")

# Non-ML references evaluated on exactly the same untouched full candidate pool.
popularity = pre_valid_purchases[COUPON_ID].astype(str).value_counts()
baseline_scores = {
    "GlobalPopularity": np.log1p(valid_raw[COUPON_ID].astype(str).map(popularity).fillna(0).to_numpy(dtype=float)),
    "UserPreference": (
        np.log1p(valid_raw["user_genre_purchase_count"].to_numpy(dtype=float)) +
        0.50 * np.log1p(valid_raw["user_genre_view_count"].to_numpy(dtype=float)) +
        0.25 * valid_raw["same_prefecture"].to_numpy(dtype=float)
    ),
}
evaluation_ap = {name: artifact["per_user_ap"] for name, artifact in model_artifacts.items()}
baseline_rows = []
for name, scores in baseline_scores.items():
    map10, _, per_user_ap = evaluate_scores(valid_raw, scores, valid_truth, TOP_K)
    evaluation_ap[name] = per_user_ap
    baseline_rows.append({
        "model": name, "kind": "baseline", "device": "-", "MAP@10": map10,
        "best_iteration": np.nan, "train_time": 0.0, "predict_time": 0.0,
    })

model_results = pd.DataFrame(experiment_rows).sort_values("MAP@10", ascending=False).reset_index(drop=True)
results = pd.concat([model_results, pd.DataFrame(baseline_rows)], ignore_index=True)
results = results.sort_values("MAP@10", ascending=False).reset_index(drop=True)
results["evaluation_split"] = "untouched_full_pool_holdout"
display(results)
ax = results.plot.bar(x="model", y="MAP@10", legend=False, figsize=(9, 4), rot=20)
ax.set_title("Rankers and Baselines vs MAP@10")
ax.set_ylabel("MAP@10")
plt.tight_layout()
plt.show()

best_model_name = model_results.iloc[0]["model"]
best_artifact = model_artifacts[best_model_name]
bootstrap_rows = []
for challenger, challenger_ap in evaluation_ap.items():
    if challenger == best_model_name:
        continue
    mean_delta, ci_low, ci_high = paired_bootstrap_delta(
        evaluation_ap[best_model_name], challenger_ap, repeats=BOOTSTRAP_REPEATS,
        seed=SEED + len(bootstrap_rows),
    )
    bootstrap_rows.append({
        "reference": best_model_name, "challenger": challenger,
        "mean_AP_delta": mean_delta, "CI_2.5%": ci_low, "CI_97.5%": ci_high,
        "conclusion": "reference better" if ci_low > 0 else "statistically tied at 95% CI",
    })
bootstrap_results = pd.DataFrame(bootstrap_rows)
display(bootstrap_results)

importance_values, importance_type = model_gain_importance(best_artifact)
feature_importance = pd.DataFrame({
    "feature": best_artifact["preprocessor"]["features"], "importance": importance_values,
}).sort_values("importance", ascending=False)
display(feature_importance.head(20))
feature_importance.head(20).sort_values("importance").plot.barh(x="feature", y="importance", legend=False, figsize=(8, 6))
plt.title(f"{best_model_name} Top 20 Feature Importance ({importance_type})")
plt.tight_layout()
plt.show()
print("Best ranker by point estimate:", best_model_name)


## 14. Test Candidate Generation

**做什么：** 可选地用全部购买正事件重新训练最佳模型，并按用户批次构造 `user × test coupon`。  
**为什么：** 验证完成后应利用全部监督数据；分批推理只保留 Top-10，避免全量笛卡尔积占满内存。  
**输入：** 全量历史、测试券、提交用户。  
**输出：** 每位用户的测试候选得分 Top-10。


In [ ]:
history_visits_full = visits[visits["I_DATE"] <= purchases["I_DATE"].max()].copy()
history_full = build_history_features(positive_events, history_visits_full, coupons_train)
user_features_full = build_user_features(users, history_full)
coupon_features_test = build_coupon_features(coupons_test)

TRAIN_USER_COUNT = train_raw[USER_ID].nunique()
TUNING_USER_COUNT = tuning_raw[USER_ID].nunique()
VALID_USER_COUNT = valid_raw[USER_ID].nunique()
del tuning_raw, valid_raw, tuning_pairs, valid_pairs
model_artifacts = {best_model_name: best_artifact}
gc.collect()

final_artifact = best_artifact
if REFIT_BEST_ON_FULL_DATA:
    selected_iterations = int(best_artifact["best_iteration"])
    print(f"Refitting {best_model_name} on every purchase event for {selected_iterations} fixed iterations...")
    new_positive_events = pd.concat([tuning_purchases, valid_purchases], ignore_index=True)
    additional_negatives = sample_hard_negatives(
        new_positive_events, coupons_train, all_purchased_sets,
        n_negatives=NEGATIVE_SAMPLE_RATIO, seed=SEED + 10,
    )
    # Reuse the expensive training negatives; sample only newly added tuning/holdout events.
    full_negatives = pd.concat([train_negatives, additional_negatives], ignore_index=True)
    full_positives = positive_events[[USER_ID, COUPON_ID, "label", "positive_event_id"]].copy()
    full_positives["reference_coupon_id"] = full_positives[COUPON_ID]
    full_positives["negative_source"] = "positive"
    full_pairs = pd.concat([full_positives, full_negatives], ignore_index=True)
    full_pairs = full_pairs.sort_values([USER_ID, "label"], ascending=[True, False], kind="stable").reset_index(drop=True)
    full_raw = build_event_time_training_features(
        full_pairs, users, coupon_features_train, positive_events, history_visits_full,
    ).sort_values(USER_ID, kind="stable").reset_index(drop=True)
    del train_raw
    gc.collect()
    final_artifact = fit_ranker(
        best_model_name, full_raw, valid_frame=None, fixed_iterations=selected_iterations,
    )
    model_artifacts["final_refit"] = final_artifact
    print("Final refit rows:", len(full_raw), "positive events:", int(full_raw["label"].sum()))
    del additional_negatives, full_negatives, full_positives, full_pairs, full_raw
    gc.collect()


def generate_test_predictions(artifact, target_users, test_coupons, user_features, history, batch_size=500, top_k=10):
    """Score all test coupons in user batches and retain only per-user Top-k rows."""
    coupon_ids = test_coupons[COUPON_ID].astype(str).to_numpy()
    outputs = []
    target_users = np.asarray(target_users, dtype=object)
    for start in range(0, len(target_users), batch_size):
        batch_users = target_users[start:start + batch_size]
        pairs = pd.DataFrame({
            USER_ID: np.repeat(batch_users, len(coupon_ids)),
            COUPON_ID: np.tile(coupon_ids, len(batch_users)),
        })
        raw = build_user_coupon_features(pairs, user_features, test_coupons, history)
        scores = predict_ranker(artifact, raw)
        outputs.append(rank_predictions(raw, scores, top_k))
        del pairs, raw, scores
        gc.collect()
        if start == 0 or start + batch_size >= len(target_users):
            print(f"Predicted users {start:,}..{min(start+batch_size, len(target_users)):,}/{len(target_users):,}")
    return pd.concat(outputs, ignore_index=True)


submission_users = sample_submission[USER_ID].astype(str).to_numpy()
test_ranked = generate_test_predictions(
    final_artifact, submission_users, coupon_features_test,
    user_features_full, history_full, batch_size=USER_BATCH_SIZE, top_k=TOP_K,
)
print("Retained test ranking rows:", test_ranked.shape)
display(test_ranked.head(12))


## 15. Final Ranking

**做什么：** 按用户和模型得分排序、去重并连接 Top-10 券 ID。  
**为什么：** 官方格式要求每个用户一行、券 ID 以空格分隔。  
**输入：** 批量推理保留的 Top-10 行。  
**输出：** 用户到 `PURCHASED_COUPONS` 字符串的映射。


In [ ]:
recommendation_strings = (
    test_ranked.sort_values([USER_ID, "score"], ascending=[True, False])
    .groupby(USER_ID, observed=True)[COUPON_ID]
    .agg(lambda values: " ".join(list(dict.fromkeys(map(str, values)))[:TOP_K]))
    .rename("PURCHASED_COUPONS")
    .reset_index()
)
display(recommendation_strings.head())
print("Users ranked:", recommendation_strings[USER_ID].nunique())


## 16. Submission

**做什么：** 按官方样例用户顺序生成并保存提交文件。  
**为什么：** 保留模板顺序和列名可避免 Kaggle 格式错误。  
**输入：** `sample_submission.csv` 与最终推荐字符串。  
**输出：** `/kaggle/working/submission_v1_1.csv`。


In [ ]:
submission = sample_submission[[USER_ID]].copy()
submission[USER_ID] = submission[USER_ID].astype(str)
submission = submission.merge(recommendation_strings, on=USER_ID, how="left", validate="one_to_one")
genre_popularity = (
    positive_events.merge(coupons_train[[COUPON_ID, "GENRE_NAME"]], on=COUPON_ID, how="left")["GENRE_NAME"].value_counts()
)
fallback_rank = coupons_test[[COUPON_ID, "GENRE_NAME"]].copy()
fallback_rank["genre_popularity"] = fallback_rank["GENRE_NAME"].map(genre_popularity).fillna(0)
fallback = " ".join(fallback_rank.sort_values("genre_popularity", ascending=False)[COUPON_ID].astype(str).head(TOP_K))
missing_before_fallback = int(submission["PURCHASED_COUPONS"].isna().sum())
print("Missing recommendations before fallback:", missing_before_fallback)
if missing_before_fallback > max(1, int(0.001 * len(submission))):
    raise RuntimeError("Too many users are missing model recommendations; refusing a silent fallback.")
submission["PURCHASED_COUPONS"] = submission["PURCHASED_COUPONS"].fillna(fallback)

output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
SUBMISSION_PATH = output_dir / "submission_v1_2.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

display(submission.head())
print("Submission shape:", submission.shape)
print("Missing recommendations after fallback:", submission["PURCHASED_COUPONS"].isna().sum())
print("Saved:", SUBMISSION_PATH)
assert submission.columns.tolist() == [USER_ID, "PURCHASED_COUPONS"]
assert len(submission) == len(sample_submission)


## 17. Experiment Summary

**做什么：** 自动汇总样本、模型、最佳结果、提交路径与重要特征。  
**为什么：** 让一次 “Run All” 的结论可以直接复核并留作下一版本基线。  
**输入：** 本次运行生成的统计和实验表。  
**输出：** V1.1 实验摘要。


In [ ]:
print("# V1.2 Experiment Summary")
print(f"Positive samples: {int(train_pairs['label'].sum()):,}")
print(f"Negative samples: {int((train_pairs['label'] == 0).sum()):,}")
print(f"Approximate hard-negative ratio: {(train_negatives['negative_source'] != 'L3_random_fallback').mean():.2%}")
print(f"Train users: {TRAIN_USER_COUNT:,}")
print(f"Early-stop full-pool users: {TUNING_USER_COUNT:,}")
print(f"Untouched holdout full-pool users: {VALID_USER_COUNT:,}")
for _, row in results.iterrows():
    print(f"{row['model']} [{row['kind']}] MAP@10: {row['MAP@10']:.6f}")
print(f"Best ranker by point estimate: {best_model_name}")
print(f"Selected final iterations: {best_artifact['best_iteration']}")
print(f"Submission path: {SUBMISSION_PATH}")
print("\nPaired user bootstrap (best ranker minus challenger):")
display(bootstrap_results)
print("\nTop 10 important features:")
display(feature_importance.head(10))


### V1.2 implementation notes

- Training features are strict event-time snapshots shared by every positive/negative row in the event.
- Early stopping and holdout both use complete active-coupon pools on fixed user samples.
- Model comparison includes non-ML baselines and paired user bootstrap confidence intervals.
- Final refitting reuses existing negatives and uses the selected iteration count without reusing tuning data for early stopping.
- LightGBM/XGBoost importance is reported as gain; CatBoost uses PredictionValuesChange.
